# Discovery


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_PATH = Path("/home/jupyter-iec2023se02/ml-backdoor-defense/data/0_raw/IoTID20/IoT Network Intrusion Dataset.csv")

df = pd.read_csv(DATA_PATH)
df.columns = [c.strip() for c in df.columns]
print(f"Loaded dataset: {df.shape[0]} rows × {df.shape[1]} cols")
print(f"Duplicates: {df.duplicated().sum()} | Target class distribution:")
print(df["Cat"].value_counts().to_dict())

: 

# Initial Cleaning

In [ ]:
df_clean = df.copy()
df_clean = df_clean.drop_duplicates().reset_index(drop=True)

drop_cols = [c for c in ["Flow_ID", "Src_IP", "Dst_IP", "Src_Port", "Dst_Port", "Timestamp"] if c in df_clean.columns]
df_clean = df_clean.drop(columns=drop_cols, errors="ignore")
df_clean = df_clean.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)

top_value_ratios = {col: df_clean[col].value_counts(normalize=True, dropna=False).iloc[0] for col in df_clean.columns}
constant_like_cols = [c for c in pd.Series(top_value_ratios)[lambda x: x > 0.999].index if c not in ["Label", "Cat", "Sub_Cat"]]
df_clean = df_clean.drop(columns=constant_like_cols, errors="ignore")
print(f"Cleaned: {df_clean.shape[0]} rows × {df_clean.shape[1]} cols")

# Data Splitting and Scaling

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import QuantileTransformer, MinMaxScaler

TARGET_COLUMN = "Cat"
label_columns = [col for col in ["Label", "Cat", "Sub_Cat"] if col in df_clean.columns]
y = df_clean[TARGET_COLUMN].copy()
X = df_clean.drop(columns=label_columns, errors="ignore").copy()

RANDOM_STATE = 42
TEST_SIZE = 0.15
VAL_SIZE_WITHIN_TRAIN = 0.15 / (1 - TEST_SIZE)

X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=VAL_SIZE_WITHIN_TRAIN, random_state=RANDOM_STATE, stratify=y_train_val)

scaler_pipeline = Pipeline([
    ('quantile_transformer', QuantileTransformer(output_distribution='normal', random_state=RANDOM_STATE)),
    ('minmax_scaler', MinMaxScaler()),
])

X_train_scaled = pd.DataFrame(scaler_pipeline.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_val_scaled = pd.DataFrame(scaler_pipeline.transform(X_val), columns=X_val.columns, index=X_val.index)
X_test_scaled = pd.DataFrame(scaler_pipeline.transform(X_test), columns=X_test.columns, index=X_test.index)
print(f"Split: train={X_train.shape[0]}, val={X_val.shape[0]}, test={X_test.shape[0]} | Scaled: {X_train_scaled.shape}")

# Label Encoding


In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_val_encoded = label_encoder.transform(y_val)
y_test_encoded = label_encoder.transform(y_test)
print(f"Classes: {list(label_encoder.classes_)}")

# Prepared Data Summary

In [ ]:
print(f"Data ready: {X_train_scaled.shape[1]} features, {len(label_encoder.classes_)} classes")
print("Train class distribution:")
train_dist = y_train.value_counts().sort_index()
for cls, count in train_dist.items():
    print(f"  {cls}: {count}")

# Model Training

We train a 3-layer MLP classifier with balanced cross-entropy loss to address class imbalance. The balanced class weights are computed as $w_c = \frac{n}{n_c \cdot k}$, where $n$ is total samples, $n_c$ is the count for class $c$, and $k$ is the number of classes. We employ early stopping on validation macro F1-score to prevent overfitting.

# TabNet Training on IoTID20

In [ ]:
import torch
from pytorch_tabnet.tab_model import TabNetClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

device = "cuda" if torch.cuda.is_available() else "cpu"

# Prepare data (use existing scaled data and label encoder)
X_train_scaled_tabnet = scaler_pipeline.fit_transform(X_train)
X_val_scaled_tabnet = scaler_pipeline.transform(X_val)
X_test_scaled_tabnet = scaler_pipeline.transform(X_test)

y_train_enc = label_encoder.transform(y_train)
y_val_enc = label_encoder.transform(y_val)
y_test_enc = label_encoder.transform(y_test)

# Train TabNet
print("=" * 70)
print("Training TabNet on Clean IoTID20")
print("=" * 70)

tabnet = TabNetClassifier(
    n_d=8, n_a=8, n_steps=8, gamma=1.3,
    n_independent=4, n_shared=4, epsilon=1e-15,
    virtual_batch_size=128, momentum=0.02, lambda_sparse=0.01,
    mask_type='sparsemax', learning_rate=0.001,
    seed=42, device_name=device
)

tabnet.fit(
    X_train_scaled_tabnet, y_train_enc,
    eval_set=[(X_val_scaled_tabnet, y_val_enc)],
    eval_metric=['accuracy'], max_epochs=100, patience=15,
    batch_size=1024, verbose=0
)

# Evaluate
y_pred = tabnet.predict(X_test_scaled_tabnet)
test_acc = accuracy_score(y_test_enc, y_pred)
test_f1 = f1_score(y_test_enc, y_pred, average='weighted', zero_division=0)

print(f"\n📊 Test Results:")
print(f"  Accuracy: {test_acc:.4f}")
print(f"  F1-Score: {test_f1:.4f}")
print(f"\n{classification_report(y_test_enc, y_pred, target_names=label_encoder.classes_, digits=4)}")

# Confusion matrix
cm = confusion_matrix(y_test_enc, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
plt.title('TabNet Confusion Matrix - IoTID20 Test Set')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

# Per-class accuracy
print("Per-Class Accuracy:")
for i, cls in enumerate(label_encoder.classes_):
    mask = y_test_enc == i
    if mask.sum() > 0:
        cls_acc = accuracy_score(y_test_enc[mask], y_pred[mask])
        print(f"  {cls}: {cls_acc:.4f} (n={mask.sum()})")